# Data Splitting and Leakage Risk Mitigation Strategy

## Risk Assessment Matrix: Dataset Leakage Vectors
Before writing any splitting code, we map out how standard random or unconstrained splitting strategies would introduce leakage into our model. Here is our explanation of 3 potential sources of leakage:

### Temporal Leakage
* **Relevance to our Dataset:** Our historical dataset moves linearly through time (2021 to 2025). Inflation, collective bargaining agreement changes, and cap smoothing over these years mean the financial structure of 2021 is significantly different from 2025.
* **The Leakage Threat:** If a standard split allows 2024 or 2025 data to sit in the training set while trying to predict 2022 or 2023 salaries in the validation set, we break the direction of time. A model cannot look into the future to understand past salaries.
* **Our Solution:** All data prior to 2025 is used to construct our training pools, and 2025 is kept aside.

### Group-Level Leakage
* **Relevance to our Dataset:** Our dataset contains recurring entities across multiple dimensions. The same player appears up to 5 times (once per year), and multiple players belong to the same team group within a given season.
* **The Leakage Threat:** If Breanna Stewart's 2021, 2022, and 2023 data points are in the training set, a random split might place her 2024 row in the validation set. The model would easily guess her 2024 salary based on her past salary rows rather than generalizing based on raw on-court production metrics. Furthermore, rows sharing identical team-level advanced stats would cheat if split across train/validation.
* **Our Solution:** Within our historical training pool (2021–2024), any cross-validation we perform must be structured sequentially by year or group-bounded by player identity to ensure the model generalizes effectively to unobserved contract cycles.

### Post-Outcome Leakage
* **Relevance to our Dataset:** WNBA contracts are signed and finalized prior to the start of the regular season based on trailing historical performance. However, our raw tables pair a player's contract salary with the stats they accumulated during that same season.
* **The Leakage Threat:** Training a model to map current-year performance directly to current-year salary introduces post-outcome bias. It assumes a front office knows exactly how many Win Shares a player will produce before they take the court.
* **Our Solution:** We explicitly document this as a pipeline boundary constraint. Our splitter acts as a defensive filter, and we advise downstream engineers to interpret features as leading indicators or utilize lagged performance vectors.

# Imports, Classes, Mapping Constants, and Helper Functions.

In [22]:
import os
import re
import numpy as np
import pandas as pd
from pathlib import Path

teammap = {
    "Atlanta Dream": "ATL", "Chicago Sky": "CHI", "Connecticut Sun": "CON",
    "Dallas Wings": "DAL", "Golden State Valkyries": "GSV", "Indiana Fever": "IND",
    "Las Vegas Aces": "LVA", "Los Angeles Sparks": "LAS", "Minnesota Lynx": "MIN",
    "New York Liberty": "NYL", "Phoenix Mercury": "PHO", "Seattle Storm": "SEA",
    "Washington Mystics": "WAS"
}

name_fix = {
    "Anastasiia Kosu": "Anastasiia Olairi Kosu", "Janelle SalaÃ¼n": "Janelle Salaun",
    "LeÃ¯la Lacan": "Leila Lacan", "Luisa GeiselsÃ¶der": "Luisa Geiselsoder",
    "Mamignan TourÃ©": "Mamignan Touré", "MariÃ¨me Badiane": "Marième Badiane",
    "Te-Hina PaoPao": "Te-Hina Paopao", "Sika KonÃ©": "Sika Kone"
}

def clean_col(name):
    text = str(name).strip().lower()
    text = re.sub(r"\b202\d\b", "", text) 
    text = text.strip()
    if "salary" in text:
        text = "salary"
    if "signing" in text:
        text = "signing"
        
    text = text.replace("%", "pct")
    text = re.sub(r"[^0-9a-z]+", "_", text)
    return text.strip("_")

def clean_df(df):
    df.columns = [clean_col(col) for col in df.columns]
    for col in df.select_dtypes(include=["object", "string"]).columns:
        df[col] = df[col].astype(str).str.strip().replace({"": np.nan, "—": np.nan, "nan": np.nan})
    return df

class WNBALeakageProofSplitter:
    def __init__(self, target_holdout_year=2025, year_col='year', target_col='salary'):
        self.target_holdout_year = target_holdout_year
        self.year_col = year_col
        self.target_col = target_col
        
        self.leakage_blacklist = [
            'dummy_x', 'dummy_y', 'signing_bonus', 'cap_pct', 'cap_percentage',
            'estimated_earnings', 'base_salary'
        ]

    def _clean_features(self, df):
        ignore_cols = [self.target_col] + self.leakage_blacklist
        feature_cols = [col for col in df.columns if col not in ignore_cols]
        return df[feature_cols], df[self.target_col]

    def get_final_holdout_split(self, df):
        if self.year_col not in df.columns:
            raise KeyError(f"The required chronological column '{self.year_col}' was not found in the dataframe.")
            
        historical_pool = df[df[self.year_col] < self.target_holdout_year].copy()
        holdout_pool = df[df[self.year_col] == self.target_holdout_year].copy()
        
        X_train_hist, y_train_hist = self._clean_features(historical_pool)
        X_test_2025, y_test_2025 = self._clean_features(holdout_pool)
        
        return X_train_hist, y_train_hist, X_test_2025, y_test_2025

    def generate_historical_cv_folds(self, X_train_hist, y_train_hist):
        years = X_train_hist[self.year_col].unique()
        sorted_years = sorted(years)
        
        if len(sorted_years) < 2:
            raise ValueError("Insufficient historical years to generate chronological CV folds.")

        X_train_hist = X_train_hist.reset_index(drop=True)
        
        for i in range(1, len(sorted_years)):
            train_years = sorted_years[:i]
            val_year = sorted_years[i]
            
            train_idx = X_train_hist[X_train_hist[self.year_col].isin(train_years)].index.tolist()
            val_idx = X_train_hist[X_train_hist[self.year_col] == val_year].index.tolist()
            
            yield np.array(train_idx), np.array(val_idx)

print("Dependencies and WNBALeakageProofSplitter class successfully compiled")

Dependencies and WNBALeakageProofSplitter class successfully compiled


# Load and merge multi-year datasets

In [20]:
all_seasons = []

repo = Path.cwd().parent
data_dir = repo / "data" / "raw"

for year in range(2021, 2026):
    try:
        adv = pd.read_csv(data_dir / f"{year}_advanced.csv")
        per = pd.read_csv(data_dir / f"{year}_per_game.csv")
        tot = pd.read_csv(data_dir / f"{year}_totals.csv")
        sal = pd.read_csv(data_dir / f"salary_{year}.csv")
        teamadv = pd.read_csv(data_dir / f"{year}_advanced-team.csv")
        stand = pd.read_csv(data_dir / f"{year}_wnba_standings.csv")
        
        adv, per, tot = clean_df(adv), clean_df(per), clean_df(tot)
        sal, teamadv, stand = clean_df(sal), clean_df(teamadv), clean_df(stand)
        
        sal["salary"] = pd.to_numeric(sal["salary"], errors="coerce")
        
        teamadv["team"] = teamadv["team"].str.replace("*", "", regex=False).str.strip().map(teammap)
        stand["team_name"] = stand["team_name"].str.replace("*", "", regex=False).str.strip().map(teammap)
        
        teamdf = teamadv.merge(stand, left_on="team", right_on="team_name", how="left", suffixes=("_adv", "_stand"))
        
        playerdf = per.merge(adv, on=["player", "team", "pos", "g", "mp"], how="outer")
        playerdf = playerdf.merge(tot, on=["player", "team", "pos", "g", "mp", "gs"], how="outer")
        playerdf["player"] = playerdf["player"].replace(name_fix)
        
        year_df = sal.merge(playerdf, on="player", how="inner", suffixes=("_sal", ""))
        year_df = year_df.merge(teamdf, on="team", how="left")
        
        # --- THE FIX: De-fragment the layout into a contiguous block ---
        year_df = year_df.copy()
        
        # Stamping the year column on the defragmented block is perfectly silent
        year_df['year'] = year
        
        all_seasons.append(year_df)
        print(f"Successfully integrated and verified data for the {year} season.")
        
    except FileNotFoundError as e:
        print(f"Skipping year {year}: Continuous file matrix not found ({e.filename})")

final_df = pd.concat(all_seasons, ignore_index=True)
final_df = final_df.drop(columns=['dummy_x', 'dummy_y'], errors='ignore')

Successfully integrated and verified data for the 2021 season.
Successfully integrated and verified data for the 2022 season.
Successfully integrated and verified data for the 2023 season.
Successfully integrated and verified data for the 2024 season.
Successfully integrated and verified data for the 2025 season.


# Apply splitter and conduct unit tests.

In [21]:
splitter = WNBALeakageProofSplitter(target_holdout_year=2025, year_col='year', target_col='salary')

X_train_hist, y_train_hist, X_test_2025, y_test_2025 = splitter.get_final_holdout_split(final_df)

def run_defensive_unit_tests():
    print("Splitting Unit Tests")
    
    # Test: Absolute separation of the out-of-time target year
    assert X_train_hist['year'].max() == 2024, "Fail: Training data contains years >= 2025"
    assert X_test_2025['year'].min() == 2025 and X_test_2025['year'].max() == 2025, \
        "Fail: Holdout set has non-2025 records"
    print("Pass: 2025 isolation secure")

    # Test: Blacklisted column purging check
    for blacklisted_col in splitter.leakage_blacklist:
        assert blacklisted_col not in X_train_hist.columns, f"Fail: Feature leakage detected, {blacklisted_col} left in train"
        assert blacklisted_col not in X_test_2025.columns, f"Fail: Feature leakage detected, {blacklisted_col} left in test"
    print("Pass: Feature leakage columns purged from feature matrices.")

    # Test: Chronological integrity of inner cross-validation folds
    fold_generator = splitter.generate_historical_cv_folds(X_train_hist, y_train_hist)
    for fold, (train_idx, val_idx) in enumerate(fold_generator, start=1):
        max_fold_train_year = X_train_hist.iloc[train_idx]['year'].max()
        min_fold_val_year = X_train_hist.iloc[val_idx]['year'].min()
        
        assert max_fold_train_year < min_fold_val_year, \
            f"Fail: Temporal leakage in CV Fold {fold}, future data bleeding backward"
        print(f"Pass: CV Fold {fold} chronological sequencing verified")

    print("\n Secure for modeling.")

run_defensive_unit_tests()

Splitting Unit Tests
Pass: 2025 isolation secure
Pass: Feature leakage columns purged from feature matrices.
Pass: CV Fold 1 chronological sequencing verified
Pass: CV Fold 2 chronological sequencing verified
Pass: CV Fold 3 chronological sequencing verified

 Secure for modeling.
